# 🎬 AI Shorts Factory (Free & Open Source)
**Generate Viral YouTube Shorts using 100% Free Tools**

This notebook automates the creation of faceless videos using:
*   **Groq API / HuggingFace**: For script writing and direction (Free tier).
*   **Edge-TTS**: For high-quality neural voiceovers (Free).
*   **Pexels/Pixabay**: For stock footage (Free API).
*   **Whisper**: For word-level subtitles (Open Source).
*   **MoviePy**: For video editing (Open Source).

### 🚀 How to use:
1.  Run the **Setup** cell to install dependencies.
2.  Enter your **API Keys** in the Configuration cell.
3.  Run the **Main Pipeline** with your topic.


In [ ]:
# @title 1. Install Dependencies
# Install system dependencies for MoviePy and Fonts
!apt install imagemagick fonts-liberation
!cat /etc/ImageMagick-6/policy.xml | sed 's/none/read,write/g'> /etc/ImageMagick-6/policy.xml

# Install Python libraries
!pip install moviepy edge-tts openai-whisper requests opencv-python

print("✅ Dependencies Installed!")

In [ ]:
# @title 2. Configuration & API Keys
import os

# @markdown ### API Keys
GROQ_API_KEY = "" # @param {type:"string"}
HUGGINGFACE_API_KEY = "" # @param {type:"string"}
PEXELS_API_KEY = "" # @param {type:"string"}
PIXABAY_API_KEY = "" # @param {type:"string"}

# @markdown ### Video Settings
VOICE_NAME = "en-US-ChristopherNeural" # @param ["en-US-ChristopherNeural", "en-US-AriaNeural", "en-US-GuyNeural"]
VOICE_SPEED = "+0%" # @param {type:"string"}
SUBTITLE_MODEL = "small" # @param ["tiny", "small", "medium"]
BACKGROUND_MUSIC_URL = "https://cdn.pixabay.com/download/audio/2022/05/27/audio_1808fbf07a.mp3?filename=lofi-study-112762.mp3" # @param {type:"string"}

# Export to env vars for the classes to pick up if needed, or pass directly
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['HUGGINGFACE_API_KEY'] = HUGGINGFACE_API_KEY
os.environ['PEXELS_API_KEY'] = PEXELS_API_KEY
os.environ['PIXABAY_API_KEY'] = PIXABAY_API_KEY

In [ ]:
# @title 3. Define Pipeline Logic (Classes)
import json
import os
import requests
import asyncio
import edge_tts
import random
import urllib.parse
import whisper
from moviepy.editor import *
from moviepy.video.tools.subtitles import SubtitlesClip
from moviepy.config import change_settings
change_settings({"IMAGEMAGICK_BINARY": "/usr/bin/convert"})

# --- 1. SCRIPT IMPROVER ---
class ScriptImprover:
    def __init__(self, groq_api_key=None, hf_api_key=None, model="llama3-8b-8192"):
        self.groq_key = groq_api_key
        self.hf_key = hf_api_key
        self.model = model
        self.groq_url = "https://api.groq.com/openai/v1/chat/completions"
        self.hf_url = "https://api-inference.huggingface.co/models/mistralai/Mistral-7B-Instruct-v0.2"

    def generate_plan(self, topic_or_script):
        system_prompt = (
            "You are an expert YouTube Shorts scriptwriter. "
            "Output a JSON object with this structure:\n"
            "{\n"
            "  \"title\": \"Title\",\n"
            "  \"segments\": [\n"
            "    {\n"
            "      \"text\": \"Spoken text\",\n"
            "      \"visual_keywords\": [\"keyword1\"], \n"
            "      \"mood\": \"energetic\"\n"
            "    }\n"
            "  ]\n"
            "}\n"
            "Rule: First segment must be a hook. Return ONLY valid JSON."
        )

        # Try Groq first
        if self.groq_key:
            try:
                print("🧠 Using Groq for Intelligence...")
                headers = {"Authorization": f"Bearer {self.groq_key}", "Content-Type": "application/json"}
                payload = {
                    "model": self.model,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": f"Script about: {topic_or_script}"}
                    ],
                    "temperature": 0.7
                }
                response = requests.post(self.groq_url, json=payload, headers=headers)
                response.raise_for_status()
                content = response.json()['choices'][0]['message']['content']
                return self._parse_json(content)
            except Exception as e:
                print(f"⚠️ Groq Error: {e}. Trying fallback...")
        
        # Try HuggingFace
        if self.hf_key:
            try:
                print("🧠 Using HuggingFace for Intelligence...")
                headers = {"Authorization": f"Bearer {self.hf_key}"}
                payload = {
                    "inputs": f"System: {system_prompt}\nUser: Script about {topic_or_script}\nAssistant:",
                    "parameters": {"max_new_tokens": 1024, "return_full_text": False}
                }
                response = requests.post(self.hf_url, json=payload, headers=headers)
                response.raise_for_status()
                content = response.json()[0]['generated_text']
                return self._parse_json(content)
            except Exception as e:
                print(f"⚠️ HuggingFace Error: {e}. Using local fallback.")

        return self._fallback_plan(topic_or_script)

    def _parse_json(self, content):
        try:
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]
            return json.loads(content.strip())
        except:
            print("⚠️ Failed to parse JSON from AI response. Using fallback.")
            return self._fallback_plan("Failed Generation")

    def _fallback_plan(self, text):
        return {
            "title": "Generated Short",
            "segments": [
                {
                    "text": text[:200],
                    "visual_keywords": ["abstract", "background"],
                    "mood": "neutral"
                }
            ]
        }

# --- 2. VOICE GENERATOR ---
class VoiceGenerator:
    def __init__(self, voice="en-US-ChristopherNeural", rate="+0%"):
        self.voice = voice
        self.rate = rate

    async def generate_segment_audio(self, text, output_file):
        communicate = edge_tts.Communicate(text, self.voice, rate=self.rate)
        await communicate.save(output_file)
        return output_file

    async def generate_full_audio(self, segments, output_dir="audio_segments"):
        os.makedirs(output_dir, exist_ok=True)
        results = []
        for i, segment in enumerate(segments):
            text = segment['text']
            filename = os.path.join(output_dir, f"segment_{i}.mp3")
            await self.generate_segment_audio(text, filename)
            results.append({
                "audio_file": filename,
                "text": text,
                "keywords": segment.get('visual_keywords', []),
                "mood": segment.get('mood', 'neutral')
            })
        return results

# --- 3. MEDIA FETCHER ---
class MediaFetcher:
    def __init__(self, pexels_key=None, pixabay_key=None):
        self.pexels_key = pexels_key
        self.pixabay_key = pixabay_key
        self.cache_dir = "media_cache"
        os.makedirs(self.cache_dir, exist_ok=True)

    def download_file(self, url, filename):
        path = os.path.join(self.cache_dir, filename)
        if os.path.exists(path):
            return path
        try:
            response = requests.get(url, stream=True)
            response.raise_for_status()
            with open(path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            return path
        except Exception as e:
            print(f"Error downloading {url}: {e}")
            return None

    def search_pexels_video(self, query, orientation="portrait"):
        if not self.pexels_key: return None
        headers = {"Authorization": self.pexels_key}
        url = f"https://api.pexels.com/videos/search?query={query}&orientation={orientation}&per_page=5"
        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            videos = response.json().get('videos', [])
            if not videos: return None
            video_data = random.choice(videos)
            video_files = video_data.get('video_files', [])
            target_file = video_files[0]
            for vf in video_files:
                 if vf['quality'] == 'hd' and (vf['width'] == 1080 or vf['height'] == 1920):
                     target_file = vf
                     break
            return target_file['link']
        except Exception as e:
            print(f"Pexels Search Error: {e}")
            return None

    def search_pixabay_video(self, query):
        if not self.pixabay_key: return None
        url = f"https://pixabay.com/api/videos/?key={self.pixabay_key}&q={urllib.parse.quote(query)}&video_type=film&per_page=5"
        try:
            response = requests.get(url)
            response.raise_for_status()
            hits = response.json().get('hits', [])
            if not hits: return None
            return random.choice(hits)['videos']['large']['url']
        except Exception as e:
            print(f"Pixabay Search Error: {e}")
            return None

    def get_visual_for_segment(self, keywords):
        query = " ".join(keywords[:2])
        video_url = self.search_pexels_video(query)
        if not video_url: video_url = self.search_pixabay_video(query)
        if video_url:
            ext = video_url.split('.')[-1].split('?')[0]
            if len(ext) > 4: ext = "mp4"
            filename = f"{query.replace(' ', '_')}_{random.randint(0,1000)}.{ext}"
            return self.download_file(video_url, filename)
        return None

# --- 4. CAPTION & EDITOR ---
class CaptionEngine:
    def __init__(self, model_size="small"):
        print(f"Loading Whisper model ({model_size})...")
        self.model = whisper.load_model(model_size)

    def create_karaoke_clips(self, audio_path, video_w, video_h):
        transcription = self.model.transcribe(audio_path, word_timestamps=True)
        clips = []
        font_size = 70
        # Use LiberationSans which is standard on Linux/Colab
        font = "Liberation-Sans-Bold"
        
        for segment in transcription['segments']:
            words = segment['words']
            for word_data in words:
                word_text = word_data['word'].strip().upper()
                start, end = word_data['start'], word_data['end']
                
                # Make sure duration is positive
                duration = end - start
                if duration < 0.1: duration = 0.1
                
                txt_clip = (TextClip(word_text, fontsize=font_size, font=font, 
                                     color='yellow', stroke_color='black', stroke_width=3)
                            .set_position(('center', 'center'))
                            .set_start(start)
                            .set_duration(duration))
                clips.append(txt_clip)
        return clips

class VideoCompositor:
    def __init__(self, output_res=(1080, 1920)):
        self.w, self.h = output_res

    def download_music(self, url):
        try:
            path = "background_music.mp3"
            if os.path.exists(path): return path
            response = requests.get(url)
            with open(path, 'wb') as f:
                f.write(response.content)
            return path
        except:
            return None

    def create_video(self, segments_data, caption_engine, music_url=None, output_filename="final_video.mp4"):
        final_clips = []
        
        # 1. Create Segments (Video + Audio)
        for seg in segments_data:
            audio_path = seg['audio_file']
            visual_path = seg.get('visual_path')
            audio_clip = AudioFileClip(audio_path)
            duration = audio_clip.duration
            
            if visual_path and os.path.exists(visual_path):
                video = VideoFileClip(visual_path).resize(height=self.h)
                if video.w < self.w: video = video.resize(width=self.w)
                video = video.crop(x1=video.w/2 - self.w/2, width=self.w, height=self.h)
                video = video.loop(duration=duration) if video.duration < duration else video.subclip(0, duration)
            else:
                video = ColorClip(size=(self.w, self.h), color=(0,0,0), duration=duration)
            
            video = video.set_audio(audio_clip)
            text_clips = caption_engine.create_karaoke_clips(audio_path, self.w, self.h)
            final_clips.append(CompositeVideoClip([video] + text_clips))
            
        final_video = concatenate_videoclips(final_clips)

        # 2. Add Background Music
        if music_url:
            music_path = self.download_music(music_url)
            if music_path:
                print("🎵 Adding Background Music...")
                bg_music = AudioFileClip(music_path)
                # Loop music to match video duration
                if bg_music.duration < final_video.duration:
                    bg_music = bg_music.loop(duration=final_video.duration)
                else:
                    bg_music = bg_music.subclip(0, final_video.duration)
                
                # Lower volume and composite
                bg_music = bg_music.volumex(0.1) # 10% volume
                final_audio = CompositeAudioClip([final_video.audio, bg_music])
                final_video = final_video.set_audio(final_audio)

        final_video.write_videofile(output_filename, fps=24, codec='libx264', audio_codec='aac')
        return output_filename

In [ ]:
# @title 4. Run Pipeline
# @markdown Enter the topic for your Short:
TOPIC = "The mystery of the Bermuda Triangle" # @param {type:"string"}

async def main():
    print("🎬 Starting AI Video Pipeline...")
    
    # 1. Script
    print(f"🧠 Generating Script for: {TOPIC}")
    improver = ScriptImprover(groq_api_key=GROQ_API_KEY, hf_api_key=HUGGINGFACE_API_KEY)
    plan = improver.generate_plan(TOPIC)
    print(f"📝 Title: {plan['title']}")
    
    # 2. Audio
    print("🎙️ Generating Voiceover...")
    voice_gen = VoiceGenerator(voice=VOICE_NAME, rate=VOICE_SPEED)
    segments = await voice_gen.generate_full_audio(plan['segments'])
    
    # 3. Visuals
    print("🎨 Fetching Stock Footage...")
    media_fetcher = MediaFetcher(pexels_key=PEXELS_API_KEY, pixabay_key=PIXABAY_API_KEY)
    for seg in segments:
        visual_path = media_fetcher.get_visual_for_segment(seg['keywords'])
        seg['visual_path'] = visual_path
        print(f"   - Segment: {seg['text'][:30]}... -> {visual_path}")

    # 4. Editing
    print("✂️ Editing & Captions...")
    caption_engine = CaptionEngine(model_size=SUBTITLE_MODEL)
    compositor = VideoCompositor()
    
    output_file = compositor.create_video(segments, caption_engine, music_url=BACKGROUND_MUSIC_URL)
    print(f"✅ Video Saved: {output_file}")

# Run the async main loop
import asyncio
try:
    await main()
except RuntimeError:
    # If loop is already running (e.g. in some notebook envs)
    asyncio.create_task(main())